# LinkedIn Industry Performance

In [1]:
from pathlib import Path

import altair as alt
import attaviz
import numpy as np
import pandas as pd

attaviz.enable()
alt.data_transformers.enable("vegafusion")


def find_project_root(marker="pyproject.toml"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "LinkedIn"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(exist_ok=True)

LHR_FILE = (
    DATA_PATH / "LinkedIn Hiring Rate" / "LinkedIn_LHR by Industry SA_Aug2026.xlsx"
)
LHR_VALUE = "LHR (SA)"
BASELINE = 1.0

WEST_AFRICA = ["Ghana", "Nigeria"]
COMPARATORS = ["India", "Kenya", "South Africa"]
COUNTRIES = WEST_AFRICA + COMPARATORS

SOURCE_NOTE = (
    "Source: LinkedIn Economic Graph, seasonally adjusted hiring rate "
    "(release Aug 2026).\n"
    "The index is normalised so that the 2016 monthly average equals 1.0."
)

In [2]:
def tidy_lhr(excel_file, sheet_name, names, countries=None):
    """Read one LHR sheet and return a tidy frame."""
    return (
        pd.read_excel(excel_file, sheet_name=sheet_name, header=3)
        .drop(columns=["Unnamed: 0"])
        .set_axis(names, axis="columns")
        .loc[lambda d: d["Country"].ne("Country")]
        .dropna(subset=names)
        .assign(
            Month=lambda d: pd.to_datetime(d["Month"]),
            Country=lambda d: d["Country"].str.strip(),
            **{LHR_VALUE: lambda d: pd.to_numeric(d[LHR_VALUE])},
        )
        .loc[lambda d: d["Country"].isin(countries) if countries else slice(None)]
        .sort_values(names[:-1])
        .reset_index(drop=True)
    )


def baseline_rule(data, value=BASELINE):
    """A single dashed rule at the 2016 baseline, drawn once per panel."""
    return (
        alt.Chart(data)
        .transform_aggregate(_rows="count()")
        .mark_rule(color=attaviz.REFERENCE, strokeDash=[4, 4])
        .encode(y=alt.datum(value))
    )

In [3]:
TIER_LABELS = [
    "Leading Industries",
    "Growing Industries",
    "Transitioning Industries",
    "Emerging Industries",
]

TIER_QUARTILE = {
    "Leading Industries": "1st Quartile",
    "Growing Industries": "2nd Quartile",
    "Transitioning Industries": "3rd Quartile",
    "Emerging Industries": "4th Quartile",
}


def assign_tiers(ranked_industries):
    """Split industries, already ordered best-first, into four balanced tiers."""
    industries = list(ranked_industries)
    n_tiers = min(len(TIER_LABELS), len(industries))
    if n_tiers == 0:
        return {}
    groups = np.array_split(np.array(industries, dtype=object), n_tiers)
    return {
        industry: TIER_LABELS[i] for i, group in enumerate(groups) for industry in group
    }


def industry_metrics(industry_lhr):
    """Summarise each country-industry series and label it with a tier."""
    metrics = (
        industry_lhr.sort_values("Month")
        .groupby(["Country", "Industry"])
        .agg(
            Average_LHR=(LHR_VALUE, "mean"),
            Max_LHR=(LHR_VALUE, "max"),
            Min_LHR=(LHR_VALUE, "min"),
            Volatility=(LHR_VALUE, "std"),
            Latest_LHR=(LHR_VALUE, "last"),
            First_LHR=(LHR_VALUE, "first"),
            Start_Date=("Month", "min"),
            End_Date=("Month", "max"),
        )
        .reset_index()
        .assign(Growth_Trend=lambda d: d["Latest_LHR"] - d["First_LHR"])
        .drop(columns="First_LHR")
        .sort_values(["Country", "Average_LHR"], ascending=[True, False])
    )
    tiers = (
        metrics.groupby("Country")["Industry"]
        .apply(lambda s: pd.Series(assign_tiers(s)))
        .rename("Tier")
        .reset_index()
        .rename(columns={"level_1": "Industry"})
    )
    return (
        metrics.merge(tiers, on=["Country", "Industry"], how="left")
        .assign(
            Start_Date=lambda d: d["Start_Date"].dt.strftime("%Y-%m"),
            End_Date=lambda d: d["End_Date"].dt.strftime("%Y-%m"),
            Tier=lambda d: pd.Categorical(
                d["Tier"], categories=TIER_LABELS, ordered=True
            ),
        )
        .loc[
            :,
            [
                "Country",
                "Tier",
                "Industry",
                "Average_LHR",
                "Max_LHR",
                "Min_LHR",
                "Volatility",
                "Latest_LHR",
                "Growth_Trend",
                "Start_Date",
                "End_Date",
            ],
        ]
        .sort_values(["Country", "Average_LHR"], ascending=[True, False])
        .reset_index(drop=True)
    )


def plot_industry_tiers(industry_lhr, metrics, country, width=300, height=190):
    """Industry time series for one country, one panel per quartile."""
    tiers = metrics.loc[
        metrics["Country"] == country, ["Industry", "Tier", "Average_LHR"]
    ]
    data = (
        industry_lhr.loc[industry_lhr["Country"] == country]
        .merge(tiers, on="Industry", how="inner")
        .assign(
            Tier=lambda d: d["Tier"].astype(str),
            Panel=lambda d: d["Tier"] + " (" + d["Tier"].map(TIER_QUARTILE) + ")",
            Series=lambda d: (
                d["Industry"] + " (Avg: " + d["Average_LHR"].round(2).astype(str) + ")"
            ),
        )
        .sort_values(["Tier", "Average_LHR"], ascending=[True, False])
    )
    panels = [
        f"{tier} ({TIER_QUARTILE[tier]})"
        for tier in TIER_LABELS
        if tier in set(data["Tier"])
    ]
    series_order = (
        data.drop_duplicates("Series")
        .sort_values("Average_LHR", ascending=False)["Series"]
        .tolist()
    )
    data = data.loc[:, ["Month", LHR_VALUE, "Industry", "Panel", "Series"]]
    highlight = alt.selection_point(fields=["Series"], bind="legend")

    line = (
        alt.Chart(data)
        .mark_line()
        .encode(
            x=alt.X("Month:T", title=None),
            y=alt.Y(f"{LHR_VALUE}:Q", title="Index (2016 = 1.0)"),
            color=alt.Color(
                "Series:N",
                title="Industry",
                sort=series_order,
                legend=alt.Legend(columns=2, symbolLimit=0, labelLimit=320),
            ),
            opacity=alt.when(highlight).then(alt.value(1)).otherwise(alt.value(0.15)),
            tooltip=[
                alt.Tooltip("Industry:N"),
                alt.Tooltip("Month:T", format="%b %Y"),
                alt.Tooltip(f"{LHR_VALUE}:Q", format=".2f"),
            ],
        )
        .add_params(highlight)
    )

    chart = (
        (baseline_rule(data) + line)
        .properties(width=width, height=height)
        .facet(
            facet=alt.Facet("Panel:N", title=None, sort=panels),
            columns=2,
            title=f"LinkedIn Hiring Rate by industry: {country}",
        )
        .resolve_scale(y="independent")
    )
    return attaviz.add_caption(chart, SOURCE_NOTE)

In [4]:
industry_lhr = tidy_lhr(
    LHR_FILE,
    sheet_name="2B - LHR SA by Ctry, Ind",
    names=["Month", "Country", "Industry", LHR_VALUE],
    countries=COUNTRIES,
).assign(Industry=lambda d: d["Industry"].str.strip())

industry_lhr.to_csv(PROCESSED_PATH / "lhr_industry.csv", index=False)

industry_lhr.groupby("Country").agg(
    industries=("Industry", "nunique"),
    months=("Month", "nunique"),
    rows=("Industry", "size"),
)

,industries,months,rows
Country,,,
Ghana,1,115,115
India,19,115,2185
Kenya,7,115,694
Nigeria,10,115,1150
South Africa,18,115,2070


In [5]:
metrics = industry_metrics(industry_lhr)
metrics.to_csv(PROCESSED_PATH / "lhr_industry_metrics.csv", index=False)

In [6]:
(
    metrics.groupby(["Country", "Tier"], observed=True)[
        ["Average_LHR", "Max_LHR", "Min_LHR", "Volatility", "Growth_Trend"]
    ]
    .mean()
    .round(2)
)

Average_LHR  Max_LHR  Min_LHR  \
Country      Tier                                                      
Ghana        Leading Industries               1.50     2.82     0.84   
India        Leading Industries               2.56     4.41     0.78   
             Growing Industries               1.79     2.88     0.50   
             Transitioning Industries         1.68     2.75     0.54   
             Emerging Industries              1.54     2.47     0.56   
Kenya        Leading Industries               1.30     1.75     0.78   
             Growing Industries               0.99     1.47     0.43   
             Transitioning Industries         0.93     1.63     0.33   
             Emerging Industries              0.91     1.35     0.39   
Nigeria      Leading Industries               2.59     4.83     0.90   
             Growing Industries               2.23     3.93     0.90   
             Transitioning Industries         1.81     4.41     0.73   
             Emerging Industries              1.38     2.54     0.41   
South Africa Leading Industries               1.19     1.75     0.58   
             Growing Industries               1.02     1.63     0.41   
             Transitioning Industries         0.97     1.39     0.49   
             Emerging Industries              0.90     1.32     0.33   

                                       Volatility  Growth_Trend  
Country      Tier                                                
Ghana        Leading Industries              0.38          0.31  
India        Leading Industries              1.03          1.75  
             Growing Industries              0.59          0.82  
             Transitioning Industries        0.50          0.53  
             Emerging Industries             0.42          0.37  
Kenya        Leading Industries              0.23          0.21  
             Growing Industries              0.20         -0.19  
             Transitioning Industries        0.24         -0.22  
             Emerging Industries             0.20         -0.36  
Nigeria      Leading Industries              1.10          2.63  
             Growing Industries              0.72          1.34  
             Transitioning Industries        0.68          1.21  
             Emerging Industries             0.32          0.57  
South Africa Leading Industries              0.24         -0.12  
             Growing Industries              0.21         -0.28  
             Transitioning Industries        0.19         -0.33  
             Emerging Industries             0.17         -0.37

In [7]:
plot_industry_tiers(industry_lhr, metrics, "Ghana")

alt.VConcatChart(...)

In [8]:
plot_industry_tiers(industry_lhr, metrics, "Nigeria")

alt.VConcatChart(...)

In [9]:
plot_industry_tiers(industry_lhr, metrics, "India")

alt.VConcatChart(...)

In [10]:
plot_industry_tiers(industry_lhr, metrics, "Kenya")

alt.VConcatChart(...)

In [11]:
plot_industry_tiers(industry_lhr, metrics, "South Africa")

alt.VConcatChart(...)

In [12]:
SKILLS_FILE = (
    DATA_PATH
    / "LinkedIn Skills Genome and Penetration"
    / "Skills Genome and Skills Pen 2026.xlsx"
)

SKILL_ORDER = [
    "Soft Skills",
    "Tech Skills",
    "Business Skills",
    "Disruptive Tech Skills",
    "Green Skills",
]

BENCHMARK = 1.0

SKILLS_NOTE = (
    "Source: LinkedIn Economic Graph, Skill Genome and Skills Penetration 2026."
)


def read_skills(sheet_name, header, names=None, countries=None):
    """Read one Skill Genome sheet and return a tidy frame."""
    data = pd.read_excel(SKILLS_FILE, sheet_name=sheet_name, header=header).dropna(
        axis="columns", how="all"
    )
    if names is not None:
        # Sheet 2A carries its header one row below the one openpyxl finds.
        data = data.set_axis(names, axis="columns")
    else:
        data = data.loc[
            :, [c for c in data.columns if not str(c).startswith("Unnamed")]
        ]
    return (
        data.loc[lambda d: d["Country"].ne("Country")]
        .dropna(subset=data.columns.tolist())
        .assign(
            **{
                c: lambda d, c=c: d[c].astype(str).str.strip()
                for c in data.columns
                if pd.api.types.is_string_dtype(data[c])
            }
        )
        .loc[lambda d: d["Country"].isin(countries) if countries else slice(None)]
        .reset_index(drop=True)
    )


def read_comparators():
    """The workbook's own comparator list, one row per country."""
    return (
        pd.read_excel(SKILLS_FILE, sheet_name="Ref - Country Comparators", header=3)
        .dropna(axis="columns", how="all")
        .dropna(subset=["Country"])
        .set_index("Country")
    )


def skill_scale(domain):
    """Consistent country colours across every skills chart."""
    domain = list(domain)
    return alt.Scale(domain=domain, range=[COUNTRY_COLOURS[c] for c in domain])

## Skills Distribution Analysis

In [13]:
skill_genome = read_skills(
    "2A - SGP Ctry Ind",
    header=3,
    names=["Country", "Industry", "Skill", "Skill Rank"],
    countries=COUNTRIES,
).assign(**{"Skill Rank": lambda d: pd.to_numeric(d["Skill Rank"])})

skill_genome.to_csv(PROCESSED_PATH / "skill_genome.csv", index=False)

skill_genome.groupby("Country").agg(
    industries=("Industry", "nunique"),
    skills=("Skill", "nunique"),
    rows=("Skill", "size"),
)

,industries,skills,rows
Country,,,
Ghana,20,466,600
India,20,403,600
Kenya,20,454,598
Nigeria,20,443,600
South Africa,20,463,600


In [14]:
TOP_HALF = ["Leading Industries", "Growing Industries"]
BOTTOM_HALF = ["Transitioning Industries", "Emerging Industries"]

VIRIDIS = [
    "#440154",
    "#482878",
    "#3e4989",
    "#31688e",
    "#26828e",
    "#1f9e89",
    "#35b779",
    "#6ece58",
    "#b5de2b",
    "#fde725",
    "#addc30",
    "#5ec962",
]
UNIQUE_FILL = "#eaf2f8"  # skill appears in one industry only


def half_industries(country, tiers):
    """Industries for one country whose LHR tier falls in the given half."""
    return metrics.loc[
        metrics["Country"].eq(country) & metrics["Tier"].isin(tiers), "Industry"
    ].tolist()


def skill_grid(country, industries, caption):
    """Industry-by-rank grid of top skills, shaded by how often each recurs."""
    subset = skill_genome.loc[
        skill_genome["Country"].eq(country) & skill_genome["Industry"].isin(industries)
    ]
    if subset.empty:
        return f"{country} — {caption}: no industries in this half."

    grid = (
        subset.sort_values(["Industry", "Skill Rank"])
        .pivot(index="Skill Rank", columns="Industry", values="Skill")
        .rename_axis(index="Rank", columns=None)
    )

    counts = subset.groupby("Skill")["Industry"].nunique()
    recurring = list(counts[counts > 1].index)
    colours = {s: VIRIDIS[i % len(VIRIDIS)] for i, s in enumerate(recurring)}

    def shade(skill):
        if pd.isna(skill):
            return ""
        if skill in colours:
            return f"background-color: {colours[skill]}; color: white"
        return f"background-color: {UNIQUE_FILL}"

    return (
        grid.style.map(shade)
        .set_caption(f"{caption} in {country}")
        .set_properties(
            **{"font-size": "11px", "padding": "4px 6px", "border": "1px solid white"}
        )
        .set_table_styles([{"selector": "th", "props": [("font-size", "11px")]}])
    )

In [15]:
display(
    skill_grid(
        "Ghana", half_industries("Ghana", TOP_HALF), "leading and growing industries"
    )
)
display(
    skill_grid(
        "Ghana",
        half_industries("Ghana", BOTTOM_HALF),
        "transitioning and emerging industries",
    )
)

,Professional Services
Rank,
1,Graphic Design
2,Ghana
3,Python (Programming Language)
4,Web Development
5,Data Entry
6,Tally ERP
7,Advertising
8,JavaScript
9,Front-End Development


'Ghana — transitioning and emerging industries: no industries in this half.'

In [16]:
display(
    skill_grid(
        "Nigeria",
        half_industries("Nigeria", TOP_HALF),
        "leading and growing industries",
    )
)
display(
    skill_grid(
        "Nigeria",
        half_industries("Nigeria", BOTTOM_HALF),
        "transitioning and emerging industries",
    )
)

,Consumer Services,Education,Financial Services,Hospitals and Health Care,Professional Services,"Technology, Information and Media"
Rank,,,,,,
1,Hausa,University Lecturing,Retail Banking,Healthcare,Virtual Assistance,Virtual Assistance
2,Virtual Assistance,Virtual Assistance,Banking,Basic Life Support (BLS),Virtual Administrative Support,React.js
3,Community Outreach,Classroom Management,Credit Risk Management,Healthcare Management,Search Engine Optimization (SEO),Virtual Administrative Support
4,Community Development,Teaching,Commercial Banking,Clinical Research,Legal Research,Search Engine Optimization (SEO)
5,Community Engagement,Virtual Administrative Support,Credit Analysis,Nursing,Graphic Design,Front-End Development
6,Capacity Building,Curriculum Development,Credit,Primary Care Nursing,User Interface Design,Web Content Writing
7,Public Health,Higher Education Teaching,Loans,Public Health,Email Management,Responsive Web Design
8,Preaching,Higher Education,Business Relationship Management,Medication Administration,Legal Writing,Creative Writing
9,Volunteering,University Teaching,Customer Service Representatives,Patient Safety,User Experience Design (UED),User Interface Design


,Administrative and Support Services,Government Administration,Manufacturing,"Oil, Gas, and Mining"
Rank,,,,
1,Virtual Assistance,Hausa,Fast-Moving Consumer Goods (FMCG),Oil and Gas
2,Virtual Administrative Support,Virtual Assistance,Good Manufacturing Practice (GMP),Petroleum
3,Ghostwriting,Classroom Management,Manufacturing,Onshore Oil and Gas Operations
4,Email Management,Educational Leadership,Pharmaceutical Sales,Gas
5,Executive Calendar Management,Curriculum Development,Pharmaceutics,Upstream Oil and Gas
6,Creative Writing,Yoruba,Electrical Maintenance,Offshore Drilling
7,Web Content Writing,Teaching,Preventive Maintenance,Oil and Gas Industry
8,Google Workspace,Public Health,Electrical Engineering,Oil and Gas Drilling
9,Copywriting,Virtual Administrative Support,Oil and Gas,Petroleum Engineering


In [17]:
display(
    skill_grid(
        "India", half_industries("India", TOP_HALF), "leading and growing industries"
    )
)
display(
    skill_grid(
        "India",
        half_industries("India", BOTTOM_HALF),
        "transitioning and emerging industries",
    )
)

,Construction,Consumer Services,Education,"Farming, Ranching, Forestry",Financial Services,Government Administration,Hospitals and Health Care,Professional Services,Real Estate and Equipment Rental Services,Utilities
Rank,,,,,,,,,,
1,STAAD-Pro,Hindi,Hindi,Agriculture,Retail Banking,Hindi,Healthcare,Core Java,Real Estate,Power Plants
2,Civil Engineering,Tally ERP,C (Programming Language),Organic Farming,Banking,C (Programming Language),Clinical Research,SQL,Residential Real Estate,Solar Energy
3,Site Execution,C (Programming Language),Python (Programming Language),Seed Production,Branch Banking,Tally ERP,Healthcare Management,Amazon Web Services (AWS),Commercial Real Estate,Power Generation
4,Construction Engineering,C++,Data Structures,Agribusiness,Branch Banking Operations,Python (Programming Language),Medical Billing,Tally ERP,Tally ERP,Project Commissioning
5,Site Coordination,Data Structures,C++,Crop Protection,Tally ERP,C++,Hospitals,Java,Hindi,Thermal Power Plant
6,Quantity Surveying,Python (Programming Language),Machine Learning,Agronomy,Management Information Systems (MIS),Curriculum Development,Medical Coding,Jenkins,Real Estate Development,Solar PV
7,Hindi,Goods and Services Tax (GST),Higher Education Teaching,Crop Management,Mutual Funds,Data Structures,ICD-10-CM,Spring Boot,Real Estate Transactions,Solar Power
8,Construction,Machine Learning,Core Java,Sustainable Agriculture,KYC Verification,Machine Learning,Denial Management,Manual Testing,Goods and Services Tax (GST),Hindi
9,Construction Site Management,Cascading Style Sheets (CSS),Tally ERP,Horticulture,Loans,Educational Leadership,Medicine,Jira,Site Execution,Electrical Engineering


,Accommodation and Food Services,Administrative and Support Services,Entertainment Providers,Manufacturing,"Oil, Gas, and Mining",Retail,"Technology, Information and Media","Transportation, Logistics, Supply Chain and Storage",Wholesale
Rank,,,,,,,,,
1,Hotel Management,IT Recruitment,Hindi,Hindi,Oil and Gas,Tally ERP,Core Java,Hindi,Tally ERP
2,Hospitality Management,Technical Recruiting,Fitness Training,Tally ERP,Project Commissioning,Hindi,Amazon Web Services (AWS),Tally ERP,Hindi
3,Hospitality Industry,Screening,Film,7 QC Tools,Petroleum,Apparel,Spring Boot,Freight Forwarding,Goods and Services Tax (GST)
4,Food and Beverage Operations,Sourcing,Fitness,Manufacturing,Hindi,Fashion,Java,Third-Party Logistics (3PL),Management Information Systems (MIS)
5,Pre-opening,Screening Resumes,Yoga,Manpower Handling,Piping and Instrumentation Drawing (P&ID),Visual Merchandising,SQL,Shipping,Channel Sales
6,Guest Service Management,Hindi,Film Production,Production Part Approval Process (PPAP),Oil and Gas Industry,Merchandising,Jenkins,Aviation,Vendor Management
7,Culinary Arts,Tally ERP,Entertainment,CATIA,Tally ERP,Retail,REST APIs,Airlines,SAP Materials Management (SAP MM)
8,Restaurant Management,Executive Search,Yoga Instruction,Geometric Dimensioning & Tolerancing,Onshore Oil and Gas Operations,Multi-Store Operations,C (Programming Language),Warehouse Operations,Cement
9,Food Preparation,Staffing Services,Film Direction,5S,Petrochemicals,Fashion Design,Data Structures,Management Information Systems (MIS),Tax Deducted at Source (TDS)


In [18]:
display(
    skill_grid(
        "Kenya", half_industries("Kenya", TOP_HALF), "leading and growing industries"
    )
)
display(
    skill_grid(
        "Kenya",
        half_industries("Kenya", BOTTOM_HALF),
        "transitioning and emerging industries",
    )
)

,Consumer Services,Education,Professional Services,"Technology, Information and Media"
Rank,,,,
1,Capacity Building,Swahili,Swahili,Swahili
2,Community Development,Curriculum Development,QuickBooks,Virtual Assistance
3,Resource Mobilization,Classroom Management,Virtual Assistance,React.js
4,Swahili,University Lecturing,Conveyancing,Search Engine Optimization (SEO)
5,Program Evaluation,Lesson Planning,Data Annotation,Python (Programming Language)
6,Non-Governmental Organizations (NGOs),Teaching,Legal Research,Agile Application Development
7,Community Outreach,Secondary Education,Data Entry,Editing
8,Community Engagement,Educational Leadership,Search Engine Optimization (SEO),Virtual Administrative Support
9,Fundraising,Higher Education,Certified Public Accounting,Exceeding Customer Expectations


,Financial Services,Government Administration,Manufacturing
Rank,,,
1,Retail Banking,Swahili,Swahili
2,Credit Risk Management,Curriculum Development,Good Manufacturing Practice (GMP)
3,Credit Analysis,Classroom Management,Manufacturing
4,Credit,Capacity Building,Pharmaceutical Sales
5,Banking,Educational Leadership,Pharmaceutics
6,Loans,Resource Mobilization,Fast-Moving Consumer Goods (FMCG)
7,Portfolio Management,Lesson Planning,Electrical Maintenance
8,Underwriting,Policy Analysis,Electrical Wiring
9,Financial Risk Management,Secondary Education,Electrical Troubleshooting


In [19]:
display(
    skill_grid(
        "South Africa",
        half_industries("South Africa", TOP_HALF),
        "leading and growing industries",
    )
)
display(
    skill_grid(
        "South Africa",
        half_industries("South Africa", BOTTOM_HALF),
        "transitioning and emerging industries",
    )
)

,Administrative and Support Services,Consumer Services,Education,Government Administration,Hospitals and Health Care,"Oil, Gas, and Mining",Real Estate and Equipment Rental Services,Retail,"Transportation, Logistics, Supply Chain and Storage",Utilities
Rank,,,,,,,,,,
1,Afrikaans,Afrikaans,Afrikaans,Afrikaans,Healthcare,Mining,Residential Real Estate,Afrikaans,Transportation,Power Generation
2,Pastel Accounting,Preaching,Tutoring,Electronic Data Capture (EDC),Afrikaans,Minerals,Property Management,Merchandising,Afrikaans,Power Plants
3,Leisure Travel,Electronic Data Capture (EDC),Classroom Management,Law Enforcement,Hospitals,Mineral Processing,Investment Properties,Retail,Freight,Energy
4,Security Operations,Pastel Accounting,Higher Education,Government,Basic Life Support (BLS),Underground Mining,Real Estate Transactions,Visual Merchandising,Freight Forwarding,Project Commissioning
5,Physical Security,Pastoral Care,Lesson Planning,Criminal Investigations,Nursing,Coal,Real Estate,Retail Sales,Warehouse Operations,Power Systems
6,Access Control,Community Outreach,University Lecturing,Curriculum Development,Healthcare Management,Gold,Real Estate Development,Retail Operations,Aviation,Nuclear
7,Electronic Data Capture (EDC),Pastoral Counseling,Curriculum Development,Public Policy,Clinical Research,Project Commissioning,Sellers,Store Management,Airlines,Fault Finding
8,Travel Management,Theology,Teaching,Classroom Management,Patient Safety,Base Metals,Rentals,Pastel Accounting,Air Freight,Project Engineering
9,Tourism,Fundraising,Pastel Accounting,Lesson Planning,Advanced Cardiac Life Support (ACLS),Iron Ore,Commercial Real Estate,Stock Taking,Commercial Aviation,Electrical Engineering


,Accommodation and Food Services,Construction,Entertainment Providers,Financial Services,Manufacturing,Professional Services,"Technology, Information and Media",Wholesale
Rank,,,,,,,,
1,Hospitality Industry,Construction,Afrikaans,Insurance,Afrikaans,Pastel Accounting,Afrikaans,Afrikaans
2,Catering,Construction Management,Entertainment,Financial Services,Manufacturing,Afrikaans,Telecommunications,Fast-Moving Consumer Goods (FMCG)
3,Food and Beverage Operations,Quantity Surveying,Fitness Training,Retirement Planning,Fault Finding,Pastel Partner,Broadcasting,Syspro
4,Afrikaans,Prokon,Fitness,Banking,Syspro,CaseWare Software,Wireless Technologies,Pastel Accounting
5,Hospitality Management,Contractors,Music Production,Retail Banking,Pastel Accounting,Electronic Data Capture (EDC),Voice over IP (VoIP),Pastel Partner
6,Food Preparation,Concrete,Sports Coaching,Financial Risk Management,Automotive,Civil Litigation,Software Development Life Cycle (SDLC),Manufacturing
7,Restaurant Management,Roads,Music,Wealth Management Services,Lean Manufacturing,Litigation,IT Integration,Stock Control
8,Menu Development,Earthworks,Wellness,Afrikaans,Fast-Moving Consumer Goods (FMCG),Legal Research,Television,Fault Finding
9,Culinary Arts,Civil Engineering,Music Industry,Investments,Continuous Improvement,Legal Writing,Managed Services,Pastel Evolution


## Skill Genome Pooled by Country (2017 - 2025)

### Relative Skill Group Penetration

In [20]:
comparators = read_comparators()

COUNTRY_COLOURS = dict(zip(COUNTRIES, attaviz.CATEGORICAL))

skill_penetration = read_skills("3A - SPP Ctry", header=5, countries=COUNTRIES).assign(
    **{c: lambda d, c=c: pd.to_numeric(d[c]) for c in ["Average", "Global", "Relative"]}
)

skill_penetration.to_csv(PROCESSED_PATH / "skill_penetration_country.csv", index=False)

comparators.loc[COUNTRIES]

,Quality,Comparator 1,Comparator 2,Comparator 3,Comparator 4,Comparator 5
Country,,,,,,
Ghana,Low,Egypt,India,Kenya,Morocco,South Africa
Nigeria,Use with caution,0,0,0,0,0
India,Moderate,Brazil,Indonesia,Kenya,Philippines,South Africa
Kenya,Moderate,Egypt,India,Morocco,Philippines,South Africa
South Africa,Strong,Brazil,Egypt,India,Kenya,Turkiye


In [21]:
def plot_country_penetration(data, width=420, height=280):
    """Relative skill-group penetration for the selected countries."""
    countries = [c for c in COUNTRIES if c in set(data["Country"])]
    highlight = alt.selection_point(fields=["Country"], bind="legend", name="pick")
    top = data["Relative"].max() * 1.08

    bars = (
        alt.Chart(data)
        .mark_bar()
        .encode(
            x=alt.X(
                "Skill:N",
                title=None,
                sort=SKILL_ORDER,
                axis=alt.Axis(labelAngle=-30),
            ),
            xOffset=alt.XOffset("Country:N", sort=countries),
            y=alt.Y(
                "mean(Relative):Q",
                title="Relative penetration",
                scale=alt.Scale(domain=[0, top]),
            ),
            color=alt.Color("Country:N", scale=skill_scale(countries), title=None),
            opacity=alt.when(highlight).then(alt.value(1)).otherwise(alt.value(0.25)),
            tooltip=[
                "Country:N",
                "Skill:N",
                alt.Tooltip("mean(Relative):Q", format=".2f", title="Relative"),
            ],
        )
        .add_params(highlight)
    )

    parity = (
        alt.Chart(data)
        .transform_aggregate(_rows="count()")
        .mark_rule(color=attaviz.REFERENCE, strokeDash=[4, 4])
        .encode(y=alt.datum(BENCHMARK))
    )

    chart = alt.layer(bars, parity).properties(
        width=width,
        height=height,
        title="Relative skill group penetration against the global benchmark",
    )
    return attaviz.add_caption(
        chart,
        [
            "Dashed line is parity with the global benchmark (1.0).",
            f"{SKILLS_NOTE}",
        ],
        align="left",
    )


plot_country_penetration(skill_penetration)

alt.VConcatChart(...)

## Skill Genome Pooled by Country and Industry (2017 - 2025)

The same relative penetration measure, broken out by industry. Use the dropdown
to change industry.

In [22]:
industry_penetration = read_skills(
    "3B - SPP Ctry Ind", header=5, countries=COUNTRIES
).assign(
    **{c: lambda d, c=c: pd.to_numeric(d[c]) for c in ["Average", "Global", "Relative"]}
)

industry_penetration.to_csv(
    PROCESSED_PATH / "skill_penetration_industry.csv", index=False
)

industry_penetration.groupby("Country").agg(
    industries=("Industry", "nunique"), rows=("Skill", "size")
)

,industries,rows
Country,,
Ghana,19,87
India,20,99
Kenya,19,92
Nigeria,19,94
South Africa,20,98


In [23]:
def plot_industry_penetration(data, width=420, height=280):
    """Relative penetration by skill group, with an industry dropdown."""
    countries = sorted(data["Country"].unique())
    industries = sorted(data["Industry"].unique())
    pick = alt.selection_point(
        fields=["Industry"],
        bind=alt.binding_select(options=industries, name="Industry  "),
        value=[{"Industry": industries[0]}],
    )
    top = data["Relative"].max() * 1.08

    base = alt.Chart(data).transform_filter(pick)

    bars = base.mark_bar().encode(
        x=alt.X("Skill:N", title=None, sort=SKILL_ORDER, axis=alt.Axis(labelAngle=-30)),
        xOffset=alt.XOffset("Country:N", sort=countries),
        y=alt.Y(
            "Relative:Q",
            title="Relative penetration",
            scale=alt.Scale(domain=[0, top]),
        ),
        color=alt.Color("Country:N", scale=skill_scale(countries), title=None),
        tooltip=[
            "Country:N",
            "Industry:N",
            "Skill:N",
            alt.Tooltip("Relative:Q", format=".2f"),
            alt.Tooltip("Average:Q", format=".3f", title="National"),
            alt.Tooltip("Global:Q", format=".3f", title="Global"),
        ],
    )

    parity = (
        base.transform_aggregate(_rows="count()")
        .mark_rule(color=attaviz.REFERENCE, strokeDash=[4, 4])
        .encode(y=alt.datum(BENCHMARK))
    )

    chart = (
        alt.layer(bars, parity)
        .add_params(pick)
        .properties(
            width=width,
            height=height,
            title="Relative skill group penetration by industry",
        )
    )
    return attaviz.add_caption(
        chart,
        [
            "Dashed line is parity with the global benchmark (1.0).",
            f"{SKILLS_NOTE}",
        ],
        align="left",
    )


plot_industry_penetration(industry_penetration)

alt.VConcatChart(...)

## Relative Importance in Gender Skill Profiles

**Relative Importance** measures how strongly each skill group characterises the
profile of women or men within an industry. It is built from TF-IDF scores, so it
highlights skills that are *distinctive* for a group rather than simply common.

For each gender-industry pair LinkedIn takes the top 30 characteristic skills by
TF-IDF, weights each by its score normalised across those 30, then aggregates
into broader skill groups.

A higher value means the skill group contributes more to what makes that gender's
profile distinctive in that industry. It does not measure how many people report
the skill.

In [24]:
gender_skills = read_skills(
    "3B - SPP Ctry Ind Gen", header=5, countries=COUNTRIES
).assign(
    **{
        c: lambda d, c=c: pd.to_numeric(d[c])
        for c in ["Relative Importance", "Skill Group Penetration"]
    },
    Gender=lambda d: d["Gender"].str.title(),
)

gender_skills.to_csv(PROCESSED_PATH / "gender_skill_profile.csv", index=False)

gender_skills.groupby("Country").agg(
    industries=("Industry", "nunique"), rows=("Skill", "size")
)

,industries,rows
Country,,
Ghana,18,88
India,20,96
Kenya,17,94
South Africa,20,70


In [25]:
def plot_gender_profile(data, width=200, height=170):
    """Dumbbell of female against male relative importance, per skill group."""
    wide = (
        data.pivot_table(
            index=["Country", "Industry", "Skill"],
            columns="Gender",
            values="Relative Importance",
            aggfunc="mean",
        )
        .reset_index()
        .rename_axis(columns=None)
    )

    paired = wide.dropna(subset=["Female", "Male"])

    industries = sorted(wide["Industry"].unique())
    countries = [c for c in COUNTRIES if c in set(wide["Country"])]
    default_industry = wide.groupby("Industry").size().idxmax()

    pick = alt.selection_point(
        fields=["Industry"],
        bind=alt.binding_select(options=industries, name="Industry  "),
        value=[{"Industry": default_industry}],
    )
    top = wide[["Female", "Male"]].max().max() * 1.08
    x_title = "Relative importance (TF-IDF weighted)"
    skills = [s for s in SKILL_ORDER if s in set(wide["Skill"])]

    def panel(country, first):
        y = alt.Y(
            "Skill:N",
            title=None,
            scale=alt.Scale(domain=skills),
            axis=alt.Axis(labels=first),
        )
        x_scale = alt.Scale(domain=[0, top])

        connector = (
            alt.Chart(paired.loc[paired["Country"].eq(country)])
            .transform_filter(pick)
            .mark_rule(color=attaviz.GREY_300, strokeWidth=2)
            .encode(
                y=y,
                x=alt.X("Female:Q", title=x_title, scale=x_scale),
                x2="Male:Q",
            )
        )
        dots = (
            alt.Chart(wide.loc[wide["Country"].eq(country)])
            .transform_filter(pick)
            .transform_fold(["Female", "Male"], as_=["Gender", "Relative Importance"])
            .transform_filter("isValid(datum['Relative Importance'])")
            .mark_point(size=90, filled=True, opacity=1)
            .encode(
                y=y,
                x=alt.X("Relative Importance:Q", title=x_title, scale=x_scale),
                color=alt.Color(
                    "Gender:N",
                    scale=alt.Scale(
                        domain=["Female", "Male"],
                        range=[attaviz.GENDER["female"], attaviz.GENDER["male"]],
                    ),
                    title=None,
                ),
                tooltip=[
                    "Country:N",
                    "Industry:N",
                    "Skill:N",
                    "Gender:N",
                    alt.Tooltip("Relative Importance:Q", format=".3f"),
                ],
            )
        )
        return (
            alt.layer(connector, dots)
            .add_params(pick)
            .properties(width=width, height=height, title=country)
        )

    chart = alt.hconcat(
        *(panel(c, i == 0) for i, c in enumerate(countries)), spacing=14
    ).properties(title="Relative importance of skill groups by gender")

    return attaviz.add_caption(
        chart,
        [
            "A skill group with one dot is reported for that gender only; a missing row is not reported at all. Nigeria is absent from this sheet.",
            f"{SKILLS_NOTE}",
        ],
        align="left",
    )


plot_gender_profile(gender_skills)

/var/folders/q1/wt8mfyzs73l2r5977rk_mkxm0000gn/T/ipykernel_46248/3687500046.py:94: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_gender_profile(gender_skills)


alt.VConcatChart(...)